# Validate against buoy observations

Compare the **offshore hindcast** (closest goal node) and the **spectral
metamodel** of each offshore source, with and without the wind correction (water
level from `forcing.water_level`), to buoy records. The dynamic runs are
compared in `09_Validate_Linear_Summation.ipynb`.

**Prerequisites**

1. Fitted models — `06_Validate_Metamodel.ipynb` (`mm.fit_all`)
2. Offshore sources (`07a`, `07b`) and the wind correction (`05b`)
3. Observations GeoJSON — `03_Generate_Observations_Configuration.ipynb` → `inputs/Observations/<Case>_observations.json`
4. Processed hourly buoy files in `inputs/Observations/processed/` (notebook 03).

The reconstructions are read from the cache (`tmp/reconstruct/`); fill it
beforehand on the compute node for long periods (a script calling
`mm.reconstruct` with the same arguments). Anything missing is computed here.

Comparisons request **`hs`, `tp`, `dir`, and `spr`**. Variables missing from the buoy
file or any model are skipped automatically.

The buoys validated are
the `ok` buoy sites of `outputs/output_sites_{model}.csv` (notebook 03) that have
`hs` in `period`; buoys outside the mesh or that can dry have no model output.

In [ ]:
import matplotlib.pyplot as plt
from utils import metamodel as mm
from utils import onshore_sites as obs
from utils import validation as val

## Configuration

The modelled buoys of the observations GeoJSON (`instrument_id` keys that are `ok`
output sites, notebook 03). Site labels are `buoy_{instrument_id}`. Processed hourly files (`hs`, `tp`, `dir`,
`spr` where measured) are built in notebook 03.

In [ ]:
period = slice("2014-01-01", "2018-12-31")  # within the water-level forcing
offshore_names = ["WHACS"]  # offshore_waves entries
wind_corrections = [False, True]  # True needs the wind correction (05b)
mda_n = 450  # must match a fitted mda_candidates entry
mm_set = "buoys"  # metamodel set (metamodel.sets) fitted on the buoys
primary = "Metamodel WHACS · wind off"  # estimate of the GUI summary figures

# Modelled buoys (ok output sites, notebook 03); processed hourly files
# (hs, tp, dir, spr) in inputs/Observations/processed/<instrument_id>.nc
sites = obs.list_snapwave_buoy_sites()

requested_variables = ["hs", "tp", "dir", "spr"]

print("Observations JSON:", obs.geojson_path())
print(f"period         : {period}")
print(f"offshore       : {offshore_names}, wind correction {wind_corrections}")
print(f"mda_n          : {mda_n}")
print(f"metamodel      : {mm_set}")
print(f"sites          : {sites}")
print(f"requested vars : {requested_variables}")

## 1 · Check models

In [ ]:
mm.require_models(mm_set, mda_n)

## 2 · Buoy observations

The processed hourly file of each buoy (notebook 03) in `period`, on
`(time, sites)` with CF-style attributes and the site metadata; buoys without
`hs` in `period` are skipped.

In [ ]:
buoy = val.load_buoy_observations(period, sites, variables=requested_variables)
skipped = sorted(set(sites) - set(buoy.sites.values))
sites = list(buoy.sites.values)
print(f"{len(sites)} buoys with hs in {period}; skipped (no data): {skipped}")
buoy

## 3 · Metamodel reconstructions and closest-goal offshore hindcast

The spectral metamodel for every offshore source x wind correction (read from
the cache), and the bulk offshore hindcast at each buoy's **closest goal node**.

In [ ]:
# Resources for the reconstructions below (at most 75% of this machine); the
# wind-on request is the larger one. Edit res if wanted (res.workers = ...).
res = mm.reconstruction_resources(
    period,
    mda_n,
    metamodel=mm_set,
    spectral=True,
    offshore_name=offshore_names[0],
    wl="forcing",
    save_spectra=False,
    wind_correction=True,
)

In [ ]:
metamodel = {
    f"Metamodel {name} · wind {'on' if wind else 'off'}": mm.reconstruct(
        period,
        mda_n,
        metamodel=mm_set,
        spectral=True,
        offshore_name=name,
        wl="forcing",
        save_spectra=False,
        wind_correction=wind,
        resources=res,
    )
    for name in offshore_names
    for wind in wind_corrections
}

offshore_hindcast, closest_goals = val.load_offshore_hindcast_at_buoys(
    period,
    sites,
    offshore_names=offshore_names,
    variables=requested_variables,
)

estimates = {**offshore_hindcast, **metamodel}

variables = val.resolve_comparison_variables(
    buoy,
    *estimates.values(),
    variables=requested_variables,
)
print(f"Comparing: {variables}")

## 4 · Metrics vs buoy

In [ ]:
results = val.compare_all(
    buoy,
    estimates,
    variables=variables,
    reference_label="Buoy",
)

summary = val.metrics_summary(results)
summary

Median over the buoys, per estimate and variable:

In [ ]:
summary.groupby(["label", "model"])[["n", "bias", "rmse", "r2"]].median().round(3)

## 4b · Save summary

In [ ]:
# `period` is a slice here (not a single string) — derive a clean,
# filesystem-safe label from it once; the summary PNG save cell reuses it.
period_label = period if isinstance(period, str) else f"{period.start}-{period.stop}"

summary_out = summary.copy()
summary_out.insert(0, "period", period_label)
summary_out.insert(1, "metamodel", mm_set)
summary_out.insert(2, "mda_candidate", mda_n)

out_path = val.buoy_validation_path("onshore", period_label)
out_path.parent.mkdir(parents=True, exist_ok=True)
summary_out.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

## 5 · Scatter plots

In [ ]:
for site in sites:
    for name, result in results.items():
        result.plot_scatter(site, variables)
        plt.suptitle(f"{name} vs buoy — {site} — {period}")
        plt.show()
    break

## 6 · Time series

Line plots of buoy observations vs each model over the validation period (direction as scatter).

In [ ]:
for site in sites:
    val.plot_buoy_timeseries(buoy, estimates, site, variables=variables)
    plt.suptitle(f"Buoy validation — {site} — {period}")
    plt.show()
    break

## 7 · Save per-buoy summary PNG (GUI)

Density scatter (top row, buoy vs. the `primary` estimate) + one wide Hs/Tp time series (bottom row, cropped to when the buoy actually has data) — one combined PNG per buoy, shown as-is in the GUI's Buoy Validation tab when you click that buoy's star on the map.

In [ ]:
for site in sites:
    fig = val.plot_buoy_summary(
        buoy,
        estimates[primary],
        site,
        variables=variables,
        reference_label="Buoy",
        period_label=period_label,
        estimate_label=primary,
    )
    out_path = val.buoy_summary_figure_path(site)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"saved {out_path}")